# Extract training samples
* Convert UAV classification images into a Sentinel-2 aligned and resolution raster with classification values where the sampling purity is met.

In [ ]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

# Values to edit

In [ ]:
sample_method = "sampling_2" # sampling_1 sampling_2
method_2_thresholds = [.5]#, .5] # .95 .97 .98 .99 1.0
max_cloud_covers = [5} " [1, 5, 10, 15] # percentage
tide_deltas = {1: {"hrs": 1, "mins": 00}} # {1: {"hrs": 1, "mins": 00}, 2: {"hrs": 0, "mins": 45}, 3: {"hrs": 0, "mins": 30}}
use_threshold = False
filtered = True

site_names = ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
lowtide_search_range_file = data_path / "ELF24505_satellite_day_search_range.csv"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"


In [ ]:
for tide_delta in tide_deltas.values():
    for max_cloud_cover in max_cloud_covers:
        for method_2_threshold in method_2_thresholds:

            print(f"Max cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and "
                  f"{tide_delta['mins']}mins. Sampling threshold {method_2_thresholds}.")

            for site_name in site_names:
                sampling.sample_site_uav_raster(
                    site_name=site_name,
                    training_labels_file=training_labels_file,
                    uav_folder=uav_folder,
                    method_2_threshold=method_2_threshold,
                    max_cloud_cover=max_cloud_cover,
                    low_tide_delta_hrs=tide_delta["hrs"],
                    low_tide_delta_mins=tide_delta["mins"],
                    use_threshold=use_threshold, filtered=filtered,
                )